# 🏏 IPL Data Analysis — Exploratory Data Analysis



In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('dark_background')
sns.set_palette('husl')
plt.rcParams.update({'figure.figsize':(12,5), 'axes.titlesize':14, 'font.family':'sans-serif'})
print('Libraries loaded.')

## 1. Load & Clean Data

In [ ]:
from utils.data_loader import load_matches, load_deliveries, merge_data
from utils.data_loader import compute_batsman_stats, compute_bowler_stats, compute_team_stats

matches    = load_matches()
deliveries = load_deliveries()
merged     = merge_data(matches, deliveries)

print(f'Matches:     {matches.shape}')
print(f'Deliveries:  {deliveries.shape}')
print(f'Merged:      {merged.shape}')
matches.head()

In [ ]:
# Data quality check
print('=== Matches null summary ===')
print(matches.isnull().sum())
print('\n=== Deliveries null summary ===')
print(deliveries.isnull().sum())
print('\n=== Matches dtypes ===')
print(matches.dtypes)

## 2. Basic Match Analysis

In [ ]:
# Matches per season
mps = matches.groupby('season').size().reset_index(name='matches')
fig, ax = plt.subplots()
ax.bar(mps['season'], mps['matches'], color='#f97316', alpha=0.85)
ax.set_title('Matches Per Season')
ax.set_xlabel('Season'); ax.set_ylabel('Matches')
plt.tight_layout(); plt.show()

In [ ]:
# Toss decision distribution
td = matches['toss_decision'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12,4))

axes[0].pie(td.values, labels=td.index, autopct='%1.1f%%',
            colors=['#f97316','#3b82f6'], startangle=140)
axes[0].set_title('Toss Decision')

# Toss win vs match win
tw_mw = matches[matches['toss_winner']==matches['winner']]
pct = round(len(tw_mw)/len(matches)*100, 1)
axes[1].bar(['Toss=Match Win','Toss≠Match Win'], [pct, 100-pct],
            color=['#22c55e','#ef4444'])
axes[1].set_title('Toss Winner = Match Winner %')
axes[1].set_ylabel('%')
plt.tight_layout(); plt.show()
print(f'Toss-to-Win conversion: {pct}%')

## 3. Team Analysis

In [ ]:
team_stats = compute_team_stats(matches)
print(team_stats[['team','matches_played','wins','losses','win_pct']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
ts = team_stats.sort_values('win_pct', ascending=True)
bars = ax.barh(ts['team'], ts['win_pct'], color=plt.cm.RdYlGn(
    [v/100 for v in ts['win_pct']]))
ax.set_title('Win Percentage by Team (All Time)')
ax.set_xlabel('Win %')
for bar, val in zip(bars, ts['win_pct']):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f'{val}%', va='center', fontsize=9)
plt.tight_layout(); plt.show()

## 4. Batsman Analysis

In [ ]:
bat_stats = compute_batsman_stats(deliveries)
top10_bat = bat_stats.nlargest(10, 'total_runs')
print(top10_bat[['player','total_runs','balls_faced','strike_rate','average','fours','sixes']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Top 10 by runs
axes[0].barh(top10_bat['player'][::-1], top10_bat['total_runs'][::-1],
             color='#f97316')
axes[0].set_title('Top 10 Batsmen — Total Runs')
axes[0].set_xlabel('Runs')

# Strike rate scatter
axes[1].scatter(top10_bat['total_runs'], top10_bat['strike_rate'],
                s=120, c='#3b82f6', alpha=0.8)
for _, row in top10_bat.iterrows():
    axes[1].annotate(row['player'].split()[-1],
                     (row['total_runs'], row['strike_rate']),
                     fontsize=8, textcoords='offset points', xytext=(5,5))
axes[1].set_title('Runs vs Strike Rate')
axes[1].set_xlabel('Total Runs'); axes[1].set_ylabel('Strike Rate')
plt.tight_layout(); plt.show()

## 5. Bowler Analysis

In [ ]:
bowl_stats = compute_bowler_stats(deliveries)
top10_bowl = bowl_stats[bowl_stats['wickets']>0].nlargest(10,'wickets')
print(top10_bowl[['player','wickets','overs_bowled','economy_rate','bowling_average']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

axes[0].barh(top10_bowl['player'][::-1], top10_bowl['wickets'][::-1],
             color='#22c55e')
axes[0].set_title('Top 10 Bowlers — Wickets')
axes[0].set_xlabel('Wickets')

axes[1].scatter(top10_bowl['wickets'], top10_bowl['economy_rate'],
                s=120, c='#a855f7', alpha=0.8)
for _, row in top10_bowl.iterrows():
    axes[1].annotate(row['player'].split()[-1],
                     (row['wickets'], row['economy_rate']),
                     fontsize=8, textcoords='offset points', xytext=(5,5))
axes[1].set_title('Wickets vs Economy Rate')
axes[1].set_xlabel('Wickets'); axes[1].set_ylabel('Economy')
plt.tight_layout(); plt.show()

## 6. Venue & Tournament Insights

In [ ]:
# Top venues
vc = matches['venue'].value_counts().head(10)
fig, ax = plt.subplots(figsize=(12,4))
ax.barh(vc.index[::-1], vc.values[::-1], color='#6366f1')
ax.set_title('Top 10 Venues by Matches Hosted')
ax.set_xlabel('Matches')
plt.tight_layout(); plt.show()

In [ ]:
# Batting first vs chasing
result_counts = matches['result'].value_counts()
print('Result breakdown:')
print(result_counts)

bat_first_win = len(matches[matches['result']=='runs'])
chase_win     = len(matches[matches['result']=='wickets'])
total         = bat_first_win + chase_win
print(f'\nBatting First Win%: {bat_first_win/total*100:.1f}%')
print(f'Chasing Win%:       {chase_win/total*100:.1f}%')

## 7. Correlation Heatmap

In [ ]:
numeric_cols = bat_stats[['total_runs','balls_faced','strike_rate','average','fours','sixes','matches']]
fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(numeric_cols.corr(), annot=True, fmt='.2f',
            cmap='coolwarm', ax=ax, linewidths=0.5)
ax.set_title('Batsman Stats Correlation')
plt.tight_layout(); plt.show()

## 8. Win Prediction Model

In [ ]:
from models.predictor import build_model, predict_winner

model, encoders, accuracy, report = build_model(matches)
print(f'Model Accuracy: {accuracy*100:.2f}%')
print('\nClassification Report:')
print(report)

In [ ]:
winner, conf = predict_winner(
    model, encoders,
    team1='Mumbai Indians',
    team2='Chennai Super Kings',
    toss_winner='Mumbai Indians',
    toss_decision='bat',
    venue='Wankhede Stadium, Mumbai'
)
print(f'Predicted Winner: {winner}  (Confidence: {conf}%)')

In [ ]:
from utils.t20_loader import (
    load_t20_matches,
    load_t20_batting,
    load_t20_bowling,
    load_t20_fow,
    load_t20_partnerships,
    load_players_info,
)

matches      = load_t20_matches()
batting      = load_t20_batting()
bowling      = load_t20_bowling()
fow          = load_t20_fow()
partnerships = load_t20_partnerships()
players      = load_players_info()

print(f'Matches     : {len(matches):,} rows  | cols: {list(matches.columns[:6])}...')
print(f'Batting     : {len(batting):,} rows  | cols: {list(batting.columns[:6])}...')
print(f'Bowling     : {len(bowling):,} rows  | cols: {list(bowling.columns[:6])}...')
print(f'FoW         : {len(fow):,} rows  | cols: {list(fow.columns[:6])}...')
print(f'Partnerships: {len(partnerships):,} rows  | cols: {list(partnerships.columns[:6])}...')
print(f'Players     : {len(players):,} rows  | cols: {list(players.columns[:6])}...')

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# T20I analysis helpers
from utils.t20_analysis import (
    analysis_host_countries,
    analysis_man_of_match,
    analysis_wicket_takers,
    analysis_run_scorers,
    analysis_most_sixes,
    analysis_successful_teams,
)

print('✅ All libraries imported successfully')

In [ ]:
# Quick sanity check
print('=== Matches sample ===')
display(matches.head(3))
print('=== Batting sample ===')
display(batting.head(3))
print('=== Bowling sample ===')
display(bowling.head(3))

In [ ]:
df_host, fig_host = analysis_host_countries(matches, top_n=15)
display(df_host)
fig_host.show()

In [ ]:
# Insight text
top_host = df_host.iloc[0]
print(f'🥇 Top host country  : {top_host["country"]}  ({top_host["matches"]:,} matches)')
print(f'🥈 2nd host country  : {df_host.iloc[1]["country"]}  ({df_host.iloc[1]["matches"]:,} matches)')
print(f'🥉 3rd host country  : {df_host.iloc[2]["country"]}  ({df_host.iloc[2]["matches"]:,} matches)')

In [ ]:
df_mom, fig_mom = analysis_man_of_match(matches, players, top_n=10)
display(df_mom)
fig_mom.show()

In [ ]:
top_mom = df_mom.iloc[0]
print(f'🏅 Most MOM awards: {top_mom["player"]}  ({top_mom["awards"]} awards)')

In [ ]:
df_wkt, fig_wkt = analysis_wicket_takers(bowling, players, top_n=10)
display(df_wkt)
fig_wkt.show()

In [ ]:
# Economy comparison scatter
if 'economy' in df_wkt.columns:
    fig_eco = px.scatter(
        df_wkt, x='economy', y='wickets', text='player', size='matches',
        color='economy', color_continuous_scale='RdYlGn_r',
        labels={'economy': 'Economy Rate', 'wickets': 'Total Wickets'},
        title='Wickets vs Economy — Top T20I Bowlers',
    )
    fig_eco.update_traces(textposition='top center')
    fig_eco.update_layout(
        plot_bgcolor='#161b22', paper_bgcolor='#0d1117', font_color='#e6edf3'
    )
    fig_eco.show()

In [ ]:
df_runs, fig_runs = analysis_run_scorers(batting, players, top_n=10)
display(df_runs)
fig_runs.show()

In [ ]:
# Strike rate comparison
if 'strike_rate' in df_runs.columns:
    fig_sr = px.bar(
        df_runs, x='player', y='strike_rate',
        color='strike_rate', color_continuous_scale='Oranges',
        text='strike_rate', labels={'strike_rate': 'Strike Rate', 'player': 'Batsman'},
        title='Strike Rate — Top T20I Run-Scorers',
    )
    fig_sr.update_traces(textposition='outside')
    fig_sr.update_layout(
        plot_bgcolor='#161b22', paper_bgcolor='#0d1117', font_color='#e6edf3'
    )
    fig_sr.show()

In [ ]:
df_sixes, fig_sixes = analysis_most_sixes(batting, players, top_n=10)
display(df_sixes)
fig_sixes.show()

In [ ]:
# Team-level sixes comparison
if 'team' in batting.columns:
    team_6s = (
        batting.groupby('team')['sixes'].sum()
        .reset_index().sort_values('sixes', ascending=False).head(10)
    )
    fig_t6 = px.bar(
        team_6s, x='team', y='sixes',
        color='sixes', color_continuous_scale='Greens',
        text='sixes', labels={'sixes': 'Sixes', 'team': 'Team'},
        title='Most Sixes Hit by Each Team',
    )
    fig_t6.update_traces(textposition='outside')
    fig_t6.update_layout(
        plot_bgcolor='#161b22', paper_bgcolor='#0d1117', font_color='#e6edf3'
    )
    fig_t6.show()

In [ ]:
df_teams, fig_teams = analysis_successful_teams(matches, top_n=10)
display(df_teams)
fig_teams.show()

In [ ]:
# Win % treemap
if 'win_pct' in df_teams.columns:
    fig_tree = px.treemap(
        df_teams, path=['team'], values='wins',
        color='win_pct', color_continuous_scale='RdYlGn',
        hover_data=['played', 'wins', 'win_pct'],
        title='T20I Success — Wins & Win % (Treemap)',
    )
    fig_tree.update_layout(paper_bgcolor='#0d1117', font_color='#e6edf3')
    fig_tree.show()

In [ ]:
# Highest individual T20I scores from batting card
if 'runs' in batting.columns:
    # Merge with player names
    bat_named = batting.copy()
    if 'batsman_id' in bat_named.columns and 'player_id' in players.columns:
        id_map = players.set_index('player_id')['player_name'].to_dict()
        bat_named['player'] = bat_named['batsman_id'].map(id_map).fillna(bat_named['batsman_id'].astype(str))
    elif 'batsman_name' in bat_named.columns:
        bat_named['player'] = bat_named['batsman_name']
    else:
        bat_named['player'] = bat_named.get('batsman_id', 'Unknown')

    top_scores = (
        bat_named.nlargest(10, 'runs')[['player', 'team', 'runs', 'balls', 'sixes', 'fours']]
        .reset_index(drop=True)
    )
    print('=== Top 10 Highest Individual T20I Scores ===')
    display(top_scores)

In [ ]:
# Toss decision distribution in T20Is
if 'toss_decision' in matches.columns:
    toss = matches['toss_decision'].value_counts().reset_index()
    toss.columns = ['decision', 'count']
    fig_toss = px.pie(
        toss, names='decision', values='count', hole=0.4,
        color_discrete_sequence=['#3b82f6', '#f97316'],
        title='Toss Decision Distribution in T20Is',
    )
    fig_toss.update_layout(paper_bgcolor='#0d1117', font_color='#e6edf3')
    fig_toss.show()

In [ ]:
# Year-wise T20I matches count
if 'year' in matches.columns:
    yr = matches.groupby('year').size().reset_index(name='matches')
    fig_yr = px.line(
        yr, x='year', y='matches', markers=True,
        color_discrete_sequence=['#58a6ff'],
        labels={'year': 'Year', 'matches': 'Matches Played'},
        title='T20I Matches Played per Year',
    )
    fig_yr.update_layout(
        plot_bgcolor='#161b22', paper_bgcolor='#0d1117', font_color='#e6edf3'
    )
    fig_yr.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
 
# Import our modules  (run from project root)
import sys, os
sys.path.insert(0, os.path.abspath("utils"))
from utils.odi_loader   import load_all
from utils.odi_analysis import (
    most_hosting_countries,
    top_mom_players,
    highest_wicket_takers,
    highest_run_scorers,
    most_sixes,
    most_successful_teams,
)
 
PALETTE = "viridis"
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False,
                      "axes.spines.right": False})
print("✅ Imports OK")

In [ ]:
data = load_all()
matches     = data["matches"]
batting     = data["batting"]
bowling     = data["bowling"]
fow         = data["fow"]
partnership = data["partnership"]
players     = data["players"]
 
for k, v in data.items():
    print(f"{k:12s}  →  {v.shape[0]:>7,} rows  ×  {v.shape[1]} cols")

In [ ]:
host = most_hosting_countries(matches, top_n=10)
print(host.to_string(index=False))
 
fig, ax = plt.subplots(figsize=(9, 4))
col = host.columns[0]   # 'country' or 'venue'
ax.barh(host[col][::-1], host["matches_hosted"][::-1],
        color=sns.color_palette(PALETTE, len(host)))
ax.set_xlabel("Matches Hosted")
ax.set_title("Top 10 Countries / Venues That Hosted Most ODI Matches", fontweight="bold")
for bar in ax.patches:
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f"{int(bar.get_width())}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
mom = top_mom_players(matches, players_df=players, top_n=10)
print(mom.to_string(index=False))
 
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(mom["player_name"][::-1], mom["mom_count"][::-1],
        color=sns.color_palette("magma", len(mom)))
ax.set_xlabel("Man of the Match Awards")
ax.set_title("Top 10 Players — Most MOM Awards in ODIs", fontweight="bold")
for bar in ax.patches:
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f"{int(bar.get_width())}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
wkt = highest_wicket_takers(bowling, players_df=players, top_n=10)
print(wkt.to_string(index=False))
 
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(wkt["player_name"], wkt["total_wickets"],
              color=sns.color_palette("rocket", len(wkt)))
ax.set_ylabel("Total Wickets")
ax.set_title("Top 10 Wicket Takers in ODI Cricket", fontweight="bold")
ax.set_xticklabels(wkt["player_name"], rotation=30, ha="right")
for bar, eco in zip(bars, wkt["avg_economy"]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 2,
            f"Eco:{eco}", ha="center", fontsize=7.5)
plt.tight_layout()
plt.show()

In [ ]:
runs = highest_run_scorers(batting, players_df=players, top_n=10)
print(runs.to_string(index=False))
 
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# Left: total runs
axes[0].barh(runs["player_name"][::-1], runs["total_runs"][::-1],
             color=sns.color_palette("Blues_r", len(runs)))
axes[0].set_xlabel("Total Runs")
axes[0].set_title("Highest Run Scorers — ODIs", fontweight="bold")
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 20,
                 bar.get_y() + bar.get_height()/2,
                 f"{int(bar.get_width()):,}", va="center", fontsize=8)
 
# Right: strike rate
colors = sns.color_palette("Oranges_r", len(runs))
axes[1].barh(runs["player_name"][::-1], runs["avg_sr"][::-1], color=colors)
axes[1].set_xlabel("Average Strike Rate")
axes[1].set_title("Avg Strike Rate — Top Run Scorers", fontweight="bold")
 
plt.tight_layout()
plt.show()
 

In [ ]:
sixes = most_sixes(batting, players_df=players, top_n=10)
print(sixes.to_string(index=False))
 
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(sixes["player_name"], sixes["total_sixes"],
       color=sns.color_palette("YlOrRd", len(sixes)))
ax.set_ylabel("Total Sixes")
ax.set_title("Top 10 Six Hitters in ODI Cricket", fontweight="bold")
ax.set_xticklabels(sixes["player_name"], rotation=30, ha="right")
for bar, inns in zip(ax.patches, sixes["innings"]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f"{int(bar.get_height())}\\n({inns} inn)",
            ha="center", fontsize=7.5)
plt.tight_layout()
plt.show()
 

In [ ]:
teams = most_successful_teams(matches, top_n=10)
print(teams.to_string(index=False))
 
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(teams["team"], teams["wins"],
              color=sns.color_palette("Set2", len(teams)))
ax.set_ylabel("Total Wins")
ax.set_title("Most Successful Teams in ODI Cricket", fontweight="bold")
ax.set_xticklabels(teams["team"], rotation=30, ha="right")
for bar, pct in zip(bars, teams["win_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1,
            f"{pct}%", ha="center", fontsize=8, color="black")
plt.tight_layout()
plt.show()


In [ ]:
#########################tEST

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")
 
import sys, os
sys.path.insert(0, os.path.abspath(".."))
 
from utils.test_loader import load_all_test_data
from utils.test_analysis import (
    get_matches_by_country,
    get_matches_by_venue,
    get_top_wicket_takers,
    get_top_run_scorers,
    get_team_rankings,
    get_dot_ball_by_bowler,
    get_dot_ball_by_team,
    get_top_partnerships,
    get_fow_summary,
)
 
PALETTE = "viridis"
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.figsize": (12, 5),
    "axes.titlesize": 14,
    "font.family": "sans-serif",
})
plt.style.use("dark_background")
print("✅ Test imports OK")

In [ ]:
_td          = load_all_test_data()
matches      = _td["matches"]
batting      = _td["batting"]
bowling      = _td["bowling"]
fow          = _td["fow"]
partnerships = _td["partnerships"]
players      = _td["players"]
 
for k, v in _td.items():
    status = "✅" if not v.empty else "❌ EMPTY"
    print(f"{k:15s}  {status}  →  {v.shape[0]:>7,} rows  ×  {v.shape[1]} cols")

In [ ]:
country_df = get_matches_by_country(matches)
venue_df   = get_matches_by_venue(matches)
 
print("=== Matches by Country / Team ===")
display(country_df.head(15))
 
print("\n=== Matches by Venue ===")
display(venue_df.head(15))
 
# Plot — venues
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
 
top_venues = venue_df.head(10)
axes[0].barh(
    top_venues.iloc[:, 0][::-1],
    top_venues["Matches"][::-1],
    color=sns.color_palette(PALETTE, len(top_venues)),
)
axes[0].set_xlabel("Matches")
axes[0].set_title("Top 10 Test Venues", fontweight="bold")
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 0.3,
                 bar.get_y() + bar.get_height() / 2,
                 f"{int(bar.get_width())}", va="center", fontsize=9)
 
top_countries = country_df.head(10)
axes[1].barh(
    top_countries.iloc[:, 0][::-1],
    top_countries.iloc[:, 1][::-1],
    color=sns.color_palette("magma", len(top_countries)),
)
axes[1].set_xlabel("Count")
axes[1].set_title("Top 10 Countries / Teams in Test Cricket", fontweight="bold")
for bar in axes[1].patches:
    axes[1].text(bar.get_width() + 0.3,
                 bar.get_y() + bar.get_height() / 2,
                 f"{int(bar.get_width())}", va="center", fontsize=9)
 
plt.suptitle("🌍 Test Cricket — Countries & Venues", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
wkt_df = get_top_wicket_takers(bowling, players_df=players, min_wickets=20)
print("=== Top Test Wicket Takers ===")
display(wkt_df.head(15))
 
name_col = "player_name" if "player_name" in wkt_df.columns else "Player ID"
top10w = wkt_df.head(10)
 
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
 
# Left: Total wickets
axes[0].barh(
    top10w[name_col][::-1],
    top10w["Wickets"][::-1],
    color=sns.color_palette("rocket", len(top10w)),
)
axes[0].set_xlabel("Total Wickets")
axes[0].set_title("Top 10 Test Wicket Takers", fontweight="bold")
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 0.5,
                 bar.get_y() + bar.get_height() / 2,
                 f"{int(bar.get_width())}", va="center", fontsize=9)
 
# Right: Economy scatter
axes[1].scatter(top10w["Wickets"], top10w["Economy"],
                s=120, c="#f97316", alpha=0.85)
for _, row in top10w.iterrows():
    label = str(row[name_col]).split()[-1] if pd.notna(row[name_col]) else ""
    axes[1].annotate(label,
                     (row["Wickets"], row["Economy"]),
                     fontsize=8, textcoords="offset points", xytext=(5, 5))
axes[1].set_xlabel("Total Wickets")
axes[1].set_ylabel("Economy Rate")
axes[1].set_title("Wickets vs Economy — Top Test Bowlers", fontweight="bold")
 
plt.suptitle("🎯 Test Cricket — Top Wicket Takers", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
run_df = get_top_run_scorers(batting, players_df=players, min_runs=200)
print("=== Top Test Run Scorers ===")
display(run_df.head(15))
 
name_col_b = "player_name" if "player_name" in run_df.columns else "Player ID"
top10r = run_df.head(10)
 
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
 
# Left: Total runs
axes[0].barh(
    top10r[name_col_b][::-1],
    top10r["Runs"][::-1],
    color=sns.color_palette("Blues_r", len(top10r)),
)
axes[0].set_xlabel("Total Runs")
axes[0].set_title("Top 10 Test Run Scorers", fontweight="bold")
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 10,
                 bar.get_y() + bar.get_height() / 2,
                 f"{int(bar.get_width()):,}", va="center", fontsize=9)
 
# Right: Average bar
axes[1].barh(
    top10r[name_col_b][::-1],
    top10r["Average"][::-1],
    color=sns.color_palette("Oranges_r", len(top10r)),
)
axes[1].set_xlabel("Batting Average")
axes[1].set_title("Batting Average — Top Run Scorers", fontweight="bold")
for bar in axes[1].patches:
    axes[1].text(bar.get_width() + 0.3,
                 bar.get_y() + bar.get_height() / 2,
                 f"{bar.get_width():.1f}", va="center", fontsize=9)
 
plt.suptitle("🏏 Test Cricket — Top Run Scorers", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()
 
# 100s & 50s comparison
if "100s" in run_df.columns and "50s" in run_df.columns:
    fig, ax = plt.subplots(figsize=(12, 4))
    x      = range(len(top10r))
    width  = 0.35
    ax.bar([i - width/2 for i in x], top10r["100s"], width,
           label="100s", color="#f97316", alpha=0.85)
    ax.bar([i + width/2 for i in x], top10r["50s"],  width,
           label="50s",  color="#3b82f6", alpha=0.85)
    ax.set_xticks(list(x))
    ax.set_xticklabels(top10r[name_col_b], rotation=30, ha="right")
    ax.set_ylabel("Count")
    ax.set_title("100s & 50s — Top Test Batsmen", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.show()
 

In [ ]:
team_df = get_team_rankings(matches)
print("=== Test Team Rankings ===")
display(team_df)
 
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
 
# Left: Wins
axes[0].barh(
    team_df["Team"][::-1],
    team_df["Wins"][::-1],
    color=sns.color_palette("Set2", len(team_df)),
)
axes[0].set_xlabel("Total Wins")
axes[0].set_title("Most Wins in Test Cricket", fontweight="bold")
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 0.3,
                 bar.get_y() + bar.get_height() / 2,
                 f"{int(bar.get_width())}", va="center", fontsize=9)
 
# Right: Win %
colors_wp = sns.color_palette("RdYlGn", len(team_df))
sorted_wp = team_df.sort_values("Win %", ascending=True)
axes[1].barh(sorted_wp["Team"], sorted_wp["Win %"], color=colors_wp)
axes[1].set_xlabel("Win %")
axes[1].set_title("Win % in Test Cricket", fontweight="bold")
for bar, val in zip(axes[1].patches, sorted_wp["Win %"]):
    axes[1].text(bar.get_width() + 0.3,
                 bar.get_y() + bar.get_height() / 2,
                 f"{val}%", va="center", fontsize=9)
 
plt.suptitle("🏆 Test Cricket — Team Rankings", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()
 
# Treemap — plotly
fig_tree = px.treemap(
    team_df, path=["Team"], values="Wins",
    color="Win %", color_continuous_scale="RdYlGn",
    hover_data=["Matches", "Wins", "Losses", "Draws", "Win %"],
    title="Test Cricket — Team Performance Treemap",
)
fig_tree.update_layout(paper_bgcolor="#0d1117", font_color="#e6edf3")
fig_tree.show()


In [ ]:
dot_bowler = get_dot_ball_by_bowler(bowling, players_df=players, min_balls=120)
dot_team   = get_dot_ball_by_team(bowling, min_balls=120)
 
print("=== Dot Ball % by Bowler (Top 15) ===")
display(dot_bowler.head(15))
 
print("\n=== Dot Ball % by Team ===")
display(dot_team)
 
name_col_d = "player_name" if "player_name" in dot_bowler.columns else "Player ID"
top10d = dot_bowler.head(10)
 
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
 
# Left: Dot Ball % by bowler
axes[0].barh(
    top10d[name_col_d][::-1],
    top10d["Dot Ball %"][::-1],
    color=sns.color_palette("cool", len(top10d)),
)
axes[0].set_xlabel("Dot Ball %")
axes[0].set_title("Top 10 Bowlers — Dot Ball %", fontweight="bold")
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 0.2,
                 bar.get_y() + bar.get_height() / 2,
                 f"{bar.get_width():.1f}%", va="center", fontsize=9)
 
# Right: Dot Ball % by team
axes[1].barh(
    dot_team["Team"][::-1],
    dot_team["Dot Ball %"][::-1],
    color=sns.color_palette("YlOrRd", len(dot_team)),
)
axes[1].set_xlabel("Dot Ball %")
axes[1].set_title("Team Dot Ball %", fontweight="bold")
for bar in axes[1].patches:
    axes[1].text(bar.get_width() + 0.2,
                 bar.get_y() + bar.get_height() / 2,
                 f"{bar.get_width():.1f}%", va="center", fontsize=9)
 
plt.suptitle("⚫ Test Cricket — Dot Ball Analysis", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()
 

In [ ]:
part_df = get_top_partnerships(partnerships, players_df=players, min_runs=100)
print("=== Top Test Partnerships ===")
display(part_df.head(15))
 
if not part_df.empty and "partnership runs" in part_df.columns:
    top15p = part_df.head(15).copy()
 
    # Label: Player1 & Player2 names if available
    if "Player 1" in top15p.columns and "Player 2" in top15p.columns:
        top15p["pair"] = (
            top15p["Player 1"].astype(str).str.split().str[-1]
            + " & "
            + top15p["Player 2"].astype(str).str.split().str[-1]
        )
    else:
        top15p["pair"] = (
            top15p["player1"].astype(str)
            + " & "
            + top15p["player2"].astype(str)
        )
 
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.barh(
        top15p["pair"][::-1],
        top15p["partnership runs"][::-1],
        color=sns.color_palette("plasma", len(top15p)),
    )
    ax.set_xlabel("Partnership Runs")
    ax.set_title("🤝 Top Test Partnerships", fontweight="bold")
    for bar in bars:
        ax.text(bar.get_width() + 1,
                bar.get_y() + bar.get_height() / 2,
                f"{int(bar.get_width())}", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
fow_df = get_fow_summary(fow)
print("=== Fall of Wickets Sample ===")
display(fow_df.head(15))
 
if not fow_df.empty and "wicket" in fow_df.columns and "runs" in fow_df.columns:
    avg_fow = (
        fow_df.groupby("wicket")["runs"]
        .mean()
        .reset_index()
        .rename(columns={"runs": "avg_score_at_fall"})
    )
 
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(avg_fow["wicket"], avg_fow["avg_score_at_fall"],
           color=sns.color_palette("Spectral_r", len(avg_fow)))
    ax.set_xlabel("Wicket Number")
    ax.set_ylabel("Avg Team Score at Fall")
    ax.set_title("📉 Average Score at Each Wicket Fall (Test)", fontweight="bold")
    ax.set_xticks(avg_fow["wicket"])
    for bar, val in zip(ax.patches, avg_fow["avg_score_at_fall"]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 1,
                f"{val:.0f}", ha="center", fontsize=9)
    plt.tight_layout()
    plt.show()